In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import polars as pl
import matplotlib.pyplot as plt

# Pinned data (see src/fetch_data.py), so these results do not change between runs.
_px = pd.read_csv("../data/prices.csv", index_col="date", parse_dates=True)
Y1, Y2 = _px[["EWH"]], _px[["EWZ"]]

y1 = pl.from_pandas(np.log(Y1)) 
y2 = pl.from_pandas(np.log(Y2))
spread = y1-y2
spread = spread.rename({'EWH': 'Spread'})
data = pl.concat([y1, y2, spread], how='horizontal')
data

axes = plt.subplot()
axes.plot(Y1.index, data['EWH'])
axes.plot(Y1.index, data['EWZ'])
# axes.plot(spy.index, data['Spread'])

In [2]:
def adf_test(series, **kwargs):
    # returns dict with stats
    res = adfuller(series.dropna().values, autolag=kwargs.get("autolag", "AIC"), regression=kwargs.get("regression", "c"))
    return {"stat": res[0], "pvalue": res[1], "usedlag": res[2], "nobs": res[3], "crit": res[4]}

def kpss_test(series, **kwargs):
    # returns dict with stats
    res = kpss(series.dropna().values, regression=kwargs.get("regression", "c"), nlags=kwargs.get("nlags", "auto"))
    return {"stat": res[0], "pvalue": res[1], "nlags": res[2], "crit": res[3]}

def half_life(spread):
    # estimate half-life by regressing Δs on s_{t-1}
    s = pd.Series(spread).dropna()
    s_lag = s.shift(1).dropna()
    ds = s.diff().dropna()
    # align
    s_lag = s_lag.loc[ds.index]
    if len(s_lag) < 5:
        return np.nan
    ols = sm.OLS(ds.values, sm.add_constant(s_lag.values)).fit()
    phi = ols.params[1]
    # phi is coefficient of lag in ds = a + phi * s_{t-1} + eps, relation to AR(1) slope: slope = 1 + phi
    ar1 = 1 + phi
    if ar1 <= 0 or ar1 >= 1:
        return np.inf
    hl = -np.log(2) / np.log(ar1)
    return float(hl)

def rolling_beta(y, x, window=252):
    # compute rolling OLS slope (beta)
    df = pd.concat([y, x], axis=1).dropna()
    df.columns = ["y","x"]
    betas = pd.Series(index=df.index, dtype=float)
    for i in range(window-1, len(df)):
        yw = df["y"].iloc[i-window+1:i+1]
        xw = df["x"].iloc[i-window+1:i+1]
        res = sm.OLS(yw.values, sm.add_constant(xw.values)).fit()
        betas.iloc[i] = res.params[1]
    return betas

def pair_checks(y: pd.Series, x: pd.Series, do_johansen=False, verbose=True):
    """
    Run a battery of checks for cointegration between y (dependent) and x (independent).
    Returns summary dict and diagnostics DataFrame.
    """
    # 0. Align & preprocess
    df = pd.concat({"y": y, "x": x}, axis=1).dropna()
    if df.empty:
        raise ValueError("No overlapping data between y and x after alignment.")
    yv = df["y"]
    xv = df["x"]

    out = {}
    # 1. Individual series I(1) tests
    out["adf_y_level"] = adf_test(yv, regression="c")
    out["adf_x_level"] = adf_test(xv, regression="c")
    out["kpss_y_level"] = kpss_test(yv, regression="c")
    out["kpss_x_level"] = kpss_test(xv, regression="c")

    out["adf_y_diff"] = adf_test(yv.diff().dropna(), regression="c")
    out["adf_x_diff"] = adf_test(xv.diff().dropna(), regression="c")
    out["kpss_y_diff"] = kpss_test(yv.diff().dropna(), regression="c")
    out["kpss_x_diff"] = kpss_test(xv.diff().dropna(), regression="c")

    # 2. Engle-Granger cointegration (OLS + ADF on residuals)
    X = sm.add_constant(xv.values)
    ols = sm.OLS(yv.values, X).fit()
    alpha, beta = float(ols.params[0]), float(ols.params[1])
    resid = pd.Series(ols.resid, index=df.index)
    coint_t, coint_pvalue, coint_crit = coint(yv, xv, autolag='AIC')
    out["ols_alpha"] = alpha
    out["ols_beta"] = beta
    out["engle_granger"] = {"t": float(coint_t), "pvalue": float(coint_pvalue), "crit": coint_crit}

    # residual stationarity
    out["adf_resid_n"] = adf_test(resid, regression="n")  # no constant recommended for residual ADF
    out["kpss_resid"] = kpss_test(resid, regression="c")

    # 3. Half-life of spread
    spread = yv - (alpha + beta * xv)
    out["half_life_days"] = half_life(spread)

    # 4. Rolling beta stability
    betas = rolling_beta(yv, xv, window=min(252, len(df)))
    out["rolling_beta_std"] = float(betas.dropna().std()) if betas.dropna().size>0 else np.nan
    out["rolling_beta_mean"] = float(betas.dropna().mean()) if betas.dropna().size>0 else np.nan

    # 5. optional Johansen
    if do_johansen:
        try:
            # det_order=0 (no trend), k_ar_diff = 1 as default; you may want to select more carefully
            joh = coint_johansen(df[["y","x"]].values, det_order=0, k_ar_diff=1)
            out["johansen_eig"] = joh.eig
            out["johansen_lr1"] = joh.lr1
            out["johansen_crit"] = joh.cvt
        except Exception as e:
            out["johansen_error"] = str(e)

    # 6. Diagnostics DataFrame for plotting
    diag = pd.DataFrame({
        "y": yv,
        "x": xv,
        "spread": spread,
        "resid": resid,
        "beta_roll": betas
    })

    # 7. Verdict heuristic (a starting filter)
    verdict = {}
    # Are both series non-stationary in levels? (expected for I(1) pair)
    verdict["y_level_nonstationary"] = (out["adf_y_level"]["pvalue"] > 0.05) and (out["kpss_y_level"]["pvalue"] < 0.05)
    verdict["x_level_nonstationary"] = (out["adf_x_level"]["pvalue"] > 0.05) and (out["kpss_x_level"]["pvalue"] < 0.05)
    # Engle-Granger residual stationary?
    verdict["resid_stationary"] = (out["adf_resid_n"]["pvalue"] < 0.05) and (out["kpss_resid"]["pvalue"] > 0.05)
    # Half-life reasonable?
    hl = out["half_life_days"]
    verdict["half_life_reasonable"] = (hl is not None) and (hl > 0) and (hl < 252)  # adjust upper bound as you like
    # Rolling-beta stable?
    verdict["beta_stable"] = (out["rolling_beta_std"] is not None) and (out["rolling_beta_std"] < max(0.2, abs(beta)*0.2))

    out["verdict"] = verdict
    out["diagnostics_df"] = diag
    if verbose:
        print("OLS beta =", beta, "alpha =", alpha)
        print("Engle-Granger p-value:", out["engle_granger"]["pvalue"])
        print("ADF on residuals (no constant) p-value:", out["adf_resid_n"]["pvalue"])
        print("Half-life (days):", out["half_life_days"])
        print("Rolling beta std:", out["rolling_beta_std"])
        print("Verdict summary:", verdict)

    return out
y = pd.Series(data['EWH'])
x = pd.Series(data['EWZ'])
pair_checks(y, x, do_johansen=False, verbose=True)

OLS beta = 0.5327698525372657 alpha = 1.1186256450789251
Engle-Granger p-value: 0.2775847468183043
ADF on residuals (no constant) p-value: 0.011891289648537567
Half-life (days): 104.41164091815347
Rolling beta std: 0.4989577837165797
Verdict summary: {'y_level_nonstationary': np.True_, 'x_level_nonstationary': np.True_, 'resid_stationary': np.False_, 'half_life_reasonable': True, 'beta_stable': False}


{'adf_y_level': {'stat': -1.0797254131294245,
  'pvalue': 0.7230890883737611,
  'usedlag': 0,
  'nobs': 1211,
  'crit': {'1%': np.float64(-3.435761408287299),
   '5%': np.float64(-2.863929614852828),
   '10%': np.float64(-2.568042270495956)}},
 'adf_x_level': {'stat': -1.6763359381087874,
  'pvalue': 0.44334429555456534,
  'usedlag': 15,
  'nobs': 1196,
  'crit': {'1%': np.float64(-3.435829423619109),
   '5%': np.float64(-2.863959622178626),
   '10%': np.float64(-2.5680582513898056)}},
 'kpss_y_level': {'stat': np.float64(2.2631759213612885),
  'pvalue': np.float64(0.01),
  'nlags': 20,
  'crit': {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}},
 'kpss_x_level': {'stat': np.float64(3.4314175397181863),
  'pvalue': np.float64(0.01),
  'nlags': 20,
  'crit': {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}},
 'adf_y_diff': {'stat': -33.60058052969702,
  'pvalue': 0.0,
  'usedlag': 0,
  'nobs': 1210,
  'crit': {'1%': np.float64(-3.4357658900670085),
   '5%': np.float64(-2.8